In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict
import pickle
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

In [3]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
FANTOM = DATA / "fantom_cat"
FANTOM_INITIAL = DATA_INITIAL / "fantom_cat"
PREPROCESSING = Path("../preprocessing").resolve()

MATRICES = PREPROCESSING / "matrices"
IPS_MATRICES = MATRICES / "ips_matrices"
FIBRO_MATRICES = MATRICES / "fibro_matrices"

INFO = PREPROCESSING / "info"
IPS_INFO = INFO / "ips_info"
FIBRO_INFO = INFO / "fibro_info"

FISHER = PREPROCESSING / "fisher_test"
IPS_FISHER = FISHER / "ips_fisher_exact_test"
FIBRO_FISHER = FISHER / "fibro_fisher_exact_test"

RESULTS = Path("../results")
FISHER_RESULTS = RESULTS / "fisher_test"

In [4]:
def fisher_exact_test(to_matrice):
    matrice = np.sum(np.load(to_matrice), axis=1)
    fisher_exact_res = fisher_exact([[matrice[0], matrice[1]], 
                         [matrice[2], matrice[3]]], alternative='two-sided')
    return fisher_exact_res[1]
    #return matrice

In [5]:
def reshape_df(df):
    df['combination'] = df['corr_sign'] + '_' + df['up_or_down_expr']
    # Создание сводной таблицы
    pivot_df = df.pivot_table(index=['hm', 'lnc_ens', 'aso'], 
                              columns='combination', 
                              values='p_value', 
                              aggfunc='mean').reset_index()
    pivot_df = pivot_df.rename(columns={"minus_down":"p_value-minus_down", "minus_up":"p_value-minus_up", "plus_down":"p_value-plus_down", "plus_up":"p_value-plus_up"})
    # Заполнение NaN значений нулями или другими значениями по необходимости
    pivot_df = pivot_df.fillna(1)
    return pivot_df

In [6]:
exp = "fibro"
len([to_matrice for to_matrice in (MATRICES / f"{exp}_matrices").glob('*/*')])

3256

In [9]:
exp = "fibro"
hm = "H3K27ac"
p_values = []
lncrna = []
aso = []
corr_sign = []
up_or_down_expr = []
for to_matrice in (MATRICES / f"{exp}_matrices" / hm).glob('*'):
    p_values.append(fisher_exact_test(to_matrice))
    lncrna.append(to_matrice.name.split("-")[0])
    aso.append(to_matrice.name.split("-")[1])
    corr_sign.append(to_matrice.name.split("-")[2])
    up_or_down_expr.append(to_matrice.name.split("-")[3])
df = pd.DataFrame({"hm": [hm for i in range(len(p_values))],
                   "lnc_ens": lncrna,
                   "aso": aso,
                   "p_value": p_values,
                   #"jaccard": jaccard_indexes,
                   "corr_sign": corr_sign,
                   "up_or_down_expr": up_or_down_expr
                  })

In [10]:
df

,hm,lnc_ens,aso,p_value,corr_sign,up_or_down_expr
0,H3K27ac,ENSG00000163364,ASO_G0163364_AD_01,1.000000,plus,up
1,H3K27ac,ENSG00000163364,ASO_G0163364_AD_01,1.000000,minus,up
2,H3K27ac,ENSG00000163364,ASO_G0163364_06,1.000000,plus,down
3,H3K27ac,ENSG00000163364,ASO_G0163364_06,1.000000,minus,down
4,H3K27ac,ENSG00000163364,ASO_G0163364_AD_01,0.050974,plus,down
...,...,...,...,...,...,...
374,H3K27ac,ENSG00000271270,ASO_G0271270_AD_10,1.000000,minus,down
375,H3K27ac,ENSG00000271270,ASO_G0271270_09,1.000000,plus,up
376,H3K27ac,ENSG00000271270,ASO_G0271270_09,0.362014,minus,up
377,H3K27ac,ENSG00000271270,ASO_G0271270_AD_10,1.000000,plus,up


In [33]:
["hm", "lnc_ens", "jaccard-minus_down", "jaccard-minus_up", "jaccard-plus_down", "jaccard-up", "p_value-minus_down", "p_value-minus_up", "p_value-plus_down", "p_value-plus_up"]
df['combination'] = df['corr_sign'] + '_' + df['up_or_down_expr']
# Создание сводной таблицы
pivot_df = df.pivot_table(index=['hm', 'lnc_ens', 'aso'], 
                          columns='combination', 
                          values=['p_value', 'jaccard'], 
                          aggfunc='mean').reset_index()
#pivot_df = pivot_df.rename(columns={"minus_down":"p_value_minus_down", "minus_up":"p_value_minus_up", "plus_down":"p_value_plus_down", "plus_up":"p_value_plus_up"})
# Заполнение NaN значений нулями или другими значениями по необходимости
pivot_df = pivot_df.fillna(1)
pivot_df

hm          lnc_ens             aso    jaccard            \
combination                                           minus_down  minus_up   
0            H3K4me3  ENSG00000185847     G0185847_05   1.000000  1.000000   
1            H3K4me3  ENSG00000185847     G0185847_06   1.000000  1.000000   
2            H3K4me3  ENSG00000185847     G0185847_08   1.000000  1.000000   
3            H3K4me3  ENSG00000196668     G0196668_02   0.001802  0.002132   
4            H3K4me3  ENSG00000196668     G0196668_08   0.000551  0.003056   
..               ...              ...             ...        ...       ...   
158          H3K4me3  ENSG00000272288     G0272288_09   0.000000  1.000000   
159          H3K4me3  ENSG00000272888  G0272888_AD_06   1.000000  1.000000   
160          H3K4me3  ENSG00000272888  G0272888_AD_07   1.000000  1.000000   
161          H3K4me3  ENSG00000273145     G0273145_01   1.000000  1.000000   
162          H3K4me3  ENSG00000273145     G0273145_04   1.000000  1.000000   

                                   p_value                                
combination plus_down   plus_up minus_down  minus_up plus_down   plus_up  
0            0.000000  0.000000   1.000000  1.000000  1.000000  1.000000  
1            0.000000  0.000000   1.000000  1.000000  1.000000  1.000000  
2            0.003663  0.000000   1.000000  1.000000  0.002202  1.000000  
3            0.002725  0.000550   0.535616  0.514076  0.539029  0.765765  
4            0.003880  0.002158   0.815729  0.368612  0.500021  0.546869  
..                ...       ...        ...       ...       ...       ...  
158          0.000000  1.000000   1.000000  1.000000  1.000000  1.000000  
159          0.000000  0.000000   1.000000  1.000000  1.000000  1.000000  
160          0.000000  0.000000   1.000000  1.000000  1.000000  1.000000  
161          0.000000  0.000000   1.000000  1.000000  1.000000  1.000000  
162          0.000000  0.000000   1.000000  1.000000  1.000000  1.000000  

[163 rows x 11 columns]

In [6]:
def job(hm, exp):
    p_values = []
    #jaccard_indexes = []
    lncrna = []
    aso = []
    corr_sign = []
    up_or_down_expr = []
    for to_matrice in (MATRICES / f"{exp}_matrices" / hm).glob('*'):
        p_values.append(fisher_exact_test(to_matrice))
        #jaccard_indexes.append(jaccard_index(to_matrice))
        lncrna.append(to_matrice.name.split("-")[0])
        aso.append(to_matrice.name.split("-")[1])
        corr_sign.append(to_matrice.name.split("-")[2])
        up_or_down_expr.append(to_matrice.name.split("-")[3])
    df = pd.DataFrame({"hm": [hm for i in range(len(p_values))],
                       "lnc_ens": lncrna,
                       "aso": aso,
                       "p_value": p_values,
                       #"jaccard": jaccard_indexes,
                       "corr_sign": corr_sign,
                       "up_or_down_expr": up_or_down_expr
                      })
    df = reshape_df(df)
    p_value_md_adj_bonf = multipletests(df["p_value-minus_down"], method='fdr_bh', alpha=0.05, is_sorted=False, returnsorted=False)[1]
    df['p_value-md_adj'] = p_value_md_adj_bonf

    p_value_mu_adj_bonf = multipletests(df["p_value-minus_up"], method='fdr_bh', alpha=0.05, is_sorted=False, returnsorted=False)[1]
    df['p_value-mu_adj'] = p_value_mu_adj_bonf

    p_value_pd_adj_bonf = multipletests(df["p_value-plus_down"], method='fdr_bh', alpha=0.05, is_sorted=False, returnsorted=False)[1]
    df['p_value-pd_adj'] = p_value_pd_adj_bonf

    p_value_pu_adj_bonf = multipletests(df["p_value-plus_up"], method='fdr_bh', alpha=0.05, is_sorted=False, returnsorted=False)[1]
    df['p_value-pu_adj'] = p_value_pu_adj_bonf

    
    to_save = FISHER / f"{exp}_fisher_exact_test"  / f"{hm}.tsv"
    if to_save.exists():
        print(f"Skipped: {exp} {hm}")
        return
    to_save.parent.mkdir(exist_ok=True, parents=True)
    df.to_csv(to_save, sep="\t", index=False)

In [13]:
fisher_results_fibro = pd.read_csv(FISHER / f"fibro_fisher_exact_test"  / "H3K9me3.tsv", sep="\t")
fisher_results_fibro

,hm,lnc_ens,aso,p_value-minus_down,p_value-minus_up,p_value-plus_down,p_value-plus_up,p_value-md_adj,p_value-mu_adj,p_value-pd_adj,p_value-pu_adj
0,H3K9me3,ENSG00000163364,ASO_G0163364_06,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.0,1.0
1,H3K9me3,ENSG00000163364,ASO_G0163364_AD_01,0.289468,0.000126,1.0,1.0,1.000000,0.001918,1.0,1.0
2,H3K9me3,ENSG00000204054,ASO_G0204054_01,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.0,1.0
3,H3K9me3,ENSG00000204054,ASO_G0204054_06,0.074814,1.000000,1.0,1.0,0.473819,1.000000,1.0,1.0
4,H3K9me3,ENSG00000204054,ASO_G0204054_10,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
71,H3K9me3,ENSG00000257815,ASO_G0257815_AD_03,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.0,1.0
72,H3K9me3,ENSG00000260032,ASO_G0260032_AD_01,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.0,1.0
73,H3K9me3,ENSG00000260032,ASO_G0260032_AD_06,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.0,1.0
74,H3K9me3,ENSG00000260565,ASO_G0260565_AD_06,1.000000,1.000000,1.0,1.0,1.000000,1.000000,1.0,1.0


In [14]:
fisher_results_fibro[fisher_results_fibro["lnc_ens"] == "ENSG00000229847"]

,hm,lnc_ens,aso,p_value-minus_down,p_value-minus_up,p_value-plus_down,p_value-plus_up,p_value-md_adj,p_value-mu_adj,p_value-pd_adj,p_value-pu_adj
23,H3K9me3,ENSG00000229847,ASO_G0229847_01,1.0,1.000000,1.000000,1.000000,1.0,1.000000,1.0,1.000000
24,H3K9me3,ENSG00000229847,ASO_G0229847_03,1.0,1.000000,1.000000,1.000000,1.0,1.000000,1.0,1.000000
25,H3K9me3,ENSG00000229847,ASO_G0229847_AD_02,1.0,1.000000,0.586861,1.000000,1.0,1.000000,1.0,1.000000
26,H3K9me3,ENSG00000229847,ASO_G0229847_AD_04,1.0,0.003033,0.645890,0.038791,1.0,0.038416,1.0,0.589619


In [7]:
list_of_hm = ['H3K27ac', 'H3K27me3', 'H3K36me3', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K79me2', 'H3K9ac', 'H3K9me3', 'H4K20me1', 'methylation']

In [8]:
to_save = FANTOM / "fantom_lv2-hg38-ens_name-mapping.tsv.gz"
fantom_lv2_mapping_hg38 = pd.read_csv(to_save, sep="\t", index_col="geneID_without_v")
fantom_lv2_mapping_hg38.head()

,geneName
geneID_without_v,
CATG00000000002,CATG00000000002.1
CATG00000000003,CATG00000000003.1
CATG00000000004,CATG00000000004.1
CATG00000000005,CATG00000000005.1
CATG00000000006,CATG00000000006.1


In [18]:
fantom_lv2_mapping_hg38.loc["ENSG00000249042"]

geneName    CTD-2015H6.3
Name: ENSG00000249042, dtype: object

In [9]:
fantom_lv2_mapping_hg38.shape

(123997, 1)

In [10]:
with Parallel(n_jobs=-1) as pool:
    chunks = pool(
        delayed(job)(hm=hm, exp=exp)
        for exp in ["fibro", "ips"]
        for hm in list_of_hm)

Skipped: ips methylation
Skipped: fibro H3K9me3
Skipped: fibro H3K27me3
Skipped: fibro H4K20me1
Skipped: fibro H3K4me2
Skipped: fibro H3K4me3
Skipped: fibro H3K4me1
Skipped: fibro H3K27ac
Skipped: ips H3K9ac
Skipped: ips H4K20me1
Skipped: ips H3K79me2
Skipped: fibro H3K9ac
Skipped: ips H3K36me3
Skipped: fibro H3K36me3
Skipped: ips H3K9me3
Skipped: ips H3K27ac
Skipped: ips H3K4me2
Skipped: fibro methylation
Skipped: ips H3K4me1
Skipped: ips H3K27me3
Skipped: fibro H3K79me2
Skipped: ips H3K4me3


mm => er => md
mp => ea => mu
pm => wa => pd
pp => wr => pu

In [10]:
fisher_results_filt_by_exp = {}
usecols=["hm", "lnc_ens", "aso", "p_value-md_adj", "p_value-mu_adj", "p_value-pd_adj", "p_value-pu_adj"]
for exp in ["fibro", "ips"]:
    fisher_results = [pd.read_csv(to_df, usecols=usecols, sep="\t") for to_df in (FISHER / f"{exp}_fisher_exact_test").glob("*.tsv")]
    fisher_results = pd.concat(fisher_results).reset_index(drop=True)

    mask = np.where(
        (fisher_results["p_value-md_adj"] < 0.05) |
        (fisher_results["p_value-mu_adj"] < 0.05) |
        (fisher_results["p_value-pd_adj"] < 0.05) |
        (fisher_results["p_value-pu_adj"] < 0.05),
        True,
        False
    )
    fisher_results_filt = fisher_results[mask].rename(columns={"p_value-md_adj":"er", "p_value-mu_adj":"ea", "p_value-pd_adj":"wa", "p_value-pu_adj":"wr"})
    fisher_results_filt["lnc"] = fisher_results_filt["lnc_ens"].apply(lambda x: fantom_lv2_mapping_hg38.loc[x, "geneName"])
    to_save = FISHER_RESULTS / f"{exp}-fisher-significant.tsv"
    fisher_results_filt.to_csv(to_save, sep="\t", index=False)
    fisher_results_filt_by_exp[exp] = fisher_results_filt

In [11]:
fisher_results_filt_by_exp["fibro"]

,hm,lnc_ens,aso,er,ea,wa,wr,lnc
1,H3K9me3,ENSG00000163364,ASO_G0163364_AD_01,1.000000e+00,1.917683e-03,1.000000,1.000000,AC017048.3
8,H3K9me3,ENSG00000214548,ASO_G0214548_01,6.904775e-01,6.172479e-04,0.001780,0.696110,MEG3
9,H3K9me3,ENSG00000214548,ASO_G0214548_AD_09,1.000000e+00,1.050761e-07,1.000000,0.000695,MEG3
10,H3K9me3,ENSG00000223478,ASO_G0223478_03,8.484336e-04,9.853195e-01,1.000000,1.000000,RP11-545E17.3
17,H3K9me3,ENSG00000225470,ASO_G0225470_AD_04,1.000000e+00,1.000000e+00,0.002088,1.000000,JPX
...,...,...,...,...,...,...,...,...
1055,H3K4me1,ENSG00000268895,ASO_G0268895_01,1.000000e+00,4.303351e-01,0.005312,1.000000,A1BG-AS1
1056,H3K4me1,ENSG00000268895,ASO_G0268895_03,7.864094e-04,4.142816e-01,0.000205,1.000000,A1BG-AS1
1057,H3K4me1,ENSG00000269834,ASO_G0269834_02,6.742146e-03,5.148661e-01,0.255875,1.000000,CTD-3018O17.3
1058,H3K4me1,ENSG00000269834,ASO_G0269834_AD_02,1.579246e-07,3.774202e-01,0.443793,1.000000,CTD-3018O17.3


In [8]:
fisher_results_filt_fibro = pd.read_csv(FISHER_RESULTS / f"fibro-fisher-significant.tsv", sep="\t")
fisher_results_filt_fibro

,hm,lnc_ens,aso,er,ea,wa,wr,lnc
0,H3K9me3,ENSG00000163364,ASO_G0163364_AD_01,1.000000e+00,1.917683e-03,1.000000,1.000000,AC017048.3
1,H3K9me3,ENSG00000214548,ASO_G0214548_01,6.904775e-01,6.172479e-04,0.001780,0.696110,MEG3
2,H3K9me3,ENSG00000214548,ASO_G0214548_AD_09,1.000000e+00,1.050761e-07,1.000000,0.000695,MEG3
3,H3K9me3,ENSG00000223478,ASO_G0223478_03,8.484336e-04,9.853195e-01,1.000000,1.000000,RP11-545E17.3
4,H3K9me3,ENSG00000225470,ASO_G0225470_AD_04,1.000000e+00,1.000000e+00,0.002088,1.000000,JPX
...,...,...,...,...,...,...,...,...
277,H3K4me1,ENSG00000268895,ASO_G0268895_01,1.000000e+00,4.303351e-01,0.005312,1.000000,A1BG-AS1
278,H3K4me1,ENSG00000268895,ASO_G0268895_03,7.864094e-04,4.142816e-01,0.000205,1.000000,A1BG-AS1
279,H3K4me1,ENSG00000269834,ASO_G0269834_02,6.742146e-03,5.148661e-01,0.255875,1.000000,CTD-3018O17.3
280,H3K4me1,ENSG00000269834,ASO_G0269834_AD_02,1.579246e-07,3.774202e-01,0.443793,1.000000,CTD-3018O17.3


In [10]:
ENSG00000229847_filt_fibro = fisher_results_filt_fibro[fisher_results_filt_fibro["lnc_ens"] == "ENSG00000229847"]
ENSG00000229847_filt_fibro

,hm,lnc_ens,aso,er,ea,wa,wr,lnc
8,H3K9me3,ENSG00000229847,ASO_G0229847_AD_04,1.000000e+00,3.841621e-02,1.000000e+00,0.589619,EMX2OS
54,methylation,ENSG00000229847,ASO_G0229847_AD_02,7.076174e-91,6.786944e-01,1.131655e-06,1.000000,EMX2OS
55,methylation,ENSG00000229847,ASO_G0229847_AD_04,3.368483e-97,9.323049e-09,2.510058e-07,0.000002,EMX2OS
172,H3K27ac,ENSG00000229847,ASO_G0229847_AD_02,1.000000e+00,1.000000e+00,4.454582e-04,1.000000,EMX2OS
255,H3K4me1,ENSG00000229847,ASO_G0229847_03,1.000000e+00,1.000000e+00,1.253246e-02,1.000000,EMX2OS
256,H3K4me1,ENSG00000229847,ASO_G0229847_AD_02,8.568693e-02,1.000000e+00,2.168021e-05,1.000000,EMX2OS
257,H3K4me1,ENSG00000229847,ASO_G0229847_AD_04,7.199604e-02,2.964581e-02,1.157157e-04,0.000064,EMX2OS


In [22]:
fisher_results[fisher_results["hm"] == "H3K27ac"]

,hm,lnc_ens,aso,p_value-md_adj,p_value-mu_adj,p_value-pd_adj,p_value-pu_adj
597,H3K27ac,ENSG00000196668,G0196668_02,0.393249,0.143507,6.696299e-08,0.794276
598,H3K27ac,ENSG00000196668,G0196668_08,1.000000,0.001563,8.765851e-07,0.000261
599,H3K27ac,ENSG00000203721,G0203721_01,1.000000,1.000000,1.000000e+00,1.000000
600,H3K27ac,ENSG00000203721,G0203721_05,1.000000,1.000000,1.000000e+00,1.000000
601,H3K27ac,ENSG00000212978,G0212978_03,0.639193,0.436469,1.384700e-01,1.000000
...,...,...,...,...,...,...,...
669,H3K27ac,ENSG00000266916,G0266916_04,1.000000,1.000000,1.000000e+00,1.000000
670,H3K27ac,ENSG00000268889,G0268889_05,1.000000,1.000000,9.272962e-01,1.000000
671,H3K27ac,ENSG00000269834,G0269834_AD_06,0.851802,0.151881,1.000000e+00,1.000000
672,H3K27ac,ENSG00000273145,G0273145_01,1.000000,1.000000,1.000000e+00,1.000000


In [23]:
fisher_results_filt_by_exp["fibro"]

,hm,lnc_ens,aso,er,ea,wa,wr,lnc
1,H3K9me3,ENSG00000163364,ASO_G0163364_AD_01,1.000000e+00,1.917683e-03,1.000000,1.000000,AC017048.3
8,H3K9me3,ENSG00000214548,ASO_G0214548_01,6.904775e-01,6.172479e-04,0.001780,0.696110,MEG3
9,H3K9me3,ENSG00000214548,ASO_G0214548_AD_09,1.000000e+00,1.050761e-07,1.000000,0.000695,MEG3
10,H3K9me3,ENSG00000223478,ASO_G0223478_03,8.484336e-04,9.853195e-01,1.000000,1.000000,RP11-545E17.3
17,H3K9me3,ENSG00000225470,ASO_G0225470_AD_04,1.000000e+00,1.000000e+00,0.002088,1.000000,JPX
...,...,...,...,...,...,...,...,...
1055,H3K4me1,ENSG00000268895,ASO_G0268895_01,1.000000e+00,4.303351e-01,0.005312,1.000000,A1BG-AS1
1056,H3K4me1,ENSG00000268895,ASO_G0268895_03,7.864094e-04,4.142816e-01,0.000205,1.000000,A1BG-AS1
1057,H3K4me1,ENSG00000269834,ASO_G0269834_02,6.742146e-03,5.148661e-01,0.255875,1.000000,CTD-3018O17.3
1058,H3K4me1,ENSG00000269834,ASO_G0269834_AD_02,1.579246e-07,3.774202e-01,0.443793,1.000000,CTD-3018O17.3


In [24]:
fisher_results_filt_by_exp["fibro"][fisher_results_filt_by_exp["fibro"]["lnc"] == "A1BG-AS1"]

,hm,lnc_ens,aso,er,ea,wa,wr,lnc
415,H3K36me3,ENSG00000268895,ASO_G0268895_01,1.000000e+00,1.000000e+00,3.188241e-13,1.000000,A1BG-AS1
416,H3K36me3,ENSG00000268895,ASO_G0268895_03,1.000000e+00,1.000000e+00,1.661640e-08,0.000015,A1BG-AS1
804,H3K27ac,ENSG00000268895,ASO_G0268895_01,6.780445e-23,1.732091e-02,7.703060e-56,0.000125,A1BG-AS1
805,H3K27ac,ENSG00000268895,ASO_G0268895_03,8.005907e-116,1.469022e-11,1.660879e-53,0.000003,A1BG-AS1
933,H3K4me3,ENSG00000268895,ASO_G0268895_03,2.752583e-02,1.000000e+00,1.000000e+00,1.000000,A1BG-AS1
1055,H3K4me1,ENSG00000268895,ASO_G0268895_01,1.000000e+00,4.303351e-01,5.312032e-03,1.000000,A1BG-AS1
1056,H3K4me1,ENSG00000268895,ASO_G0268895_03,7.864094e-04,4.142816e-01,2.047430e-04,1.000000,A1BG-AS1


In [12]:
ips_results_filt_fibro = pd.read_csv(FISHER_RESULTS / f"ips-fisher-significant.tsv", sep="\t")
ips_results_filt_fibro

,hm,lnc_ens,aso,er,ea,wa,wr,lnc
0,H3K4me3,ENSG00000225975,G0225975_05,1.000000e+00,1.645201e-03,1.000000e+00,1.000000e+00,AC074138.3
1,H3K4me3,ENSG00000234608,G0234608_03,2.363291e-03,1.000000e+00,5.232568e-02,1.297265e-12,MAPKAPK5-AS1
2,H3K4me3,ENSG00000234608,G0234608_AD_08,1.000000e+00,1.000000e+00,1.086221e-08,1.000000e+00,MAPKAPK5-AS1
3,H3K4me3,ENSG00000235560,G0235560_01,4.437244e-02,1.000000e+00,1.000000e+00,1.000000e+00,AC002310.12
4,H3K4me3,ENSG00000245937,G0245937_01,4.437244e-02,4.663546e-01,1.000000e+00,1.000000e+00,CTC-228N24.3
5,H3K4me3,ENSG00000247556,G0247556_09,1.255939e-08,6.754636e-10,1.000000e+00,1.000000e+00,OIP5-AS1
6,methylation,ENSG00000225484,G0225484_02,1.000000e+00,1.000000e+00,4.000629e-02,1.000000e+00,RP11-773D16.1
7,methylation,ENSG00000229153,G0229153_09,2.711532e-01,1.442290e-04,1.940174e-01,6.586324e-01,EPHA1-AS1
8,methylation,ENSG00000230844,G0230844_AD_07,1.000000e+00,1.145524e-03,7.588105e-01,7.732718e-01,ZNF674-AS1
9,H3K9ac,ENSG00000196668,G0196668_08,1.000000e+00,1.000000e+00,7.050809e-03,1.000000e+00,LINC00173


In [15]:
ips_results_filt_fibro["lnc"].unique()

array(['AC074138.3', 'MAPKAPK5-AS1', 'AC002310.12', 'CTC-228N24.3',
       'OIP5-AS1', 'RP11-773D16.1', 'EPHA1-AS1', 'ZNF674-AS1',
       'LINC00173', 'AC009299.3', 'RP11-111A21.1', 'RP3-510D11.2',
       'SBF2-AS1', 'CTD-2015H6.3', 'RP11-421L21.3', 'LINC01021',
       'AC007255.8', 'RP11-7F17.3', 'RP11-849I19.1', 'RP11-469A15.2',
       'RP11-126K1.6', 'LINC00883'], dtype=object)